# Example vertebrae with landmarks, by life history strategy

Six life history strategies in two blocks of three; each block contributes a SIDE row
and a BACK row, giving a 4 x 3 panel. Rendering matches the PC warp grid (same
renderers, material, rotation, crop).

*Last edited 18 Sep 2026 by K. Wolcott*

In [ ]:
# Imports and paths

import os, re, json, gc
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
import cv2
import pyvista as pv
import open3d as o3d
from IPython.display import Image as IPyImage

from NSM.plotting import load_mrk_json
from NSM.helper_funcs import pv_to_o3d, render_cameras

# Specify training directory and atlas directory
RUN          = "run_v72"                      # training attempt directory
ATLAS_RUN    = "2026_07-15_13_06_22/"  # atlas/builder run that produced alignedLMs
DROPBOX_ROOT = Path("/home/k.wolcott/UFL Dropbox/Katherine Wolcott/neural_shape_models/final_dataset_aug26/atlas/")
SPECIES_CSV = "../lizard_species_list.csv"

# Build other directories relative to those above
cwd      = Path.cwd()
base_wd  = cwd.parent
train_dir = base_wd / RUN
os.chdir(train_dir)
print(f"Working directory: {os.getcwd()}")

LM_DIR       = DROPBOX_ROOT / ATLAS_RUN / "alignedLMs"
OUT_DIR = Path("life_hist_exemplars")
OUT_DIR.mkdir(exist_ok=True)
OUT_PANEL = OUT_DIR / "fig_f_exemplar_lifehist_vert.png"
print(f"Outputs will be written to: {OUT_DIR.resolve()}")

# Which specimen represents each trait
EXEMPLARS = {
    "arboreal": ("chamaeleonidae_chamaeleo_calyptratus_uf-191369", "c5"),
    "burrowing": ("amphisbaenidae_bipes_biporus_uf42060", "l5"),
    "grass-swimmer": ("gerrhosauridae_tetradactylus_tetradactylus_ummz", "c6"),
    "saxicolous": ("agamidae_agama_atra_uf180711", "c5"), 
    "terrestrial": ("hoplocercidae_enyaloides_oshaughnessyi_uf191439", "c5"),
    "snake": ("homalopsidae_homalopsis_buccata_uf61845", "t10")}

# Figure bg and layout
VIEWS  = [("SIDE", "top_right",   0),
          ("BACK", "bottom_left", 180)]
N_COLS = 3
FONT        = cv2.FONT_HERSHEY_DUPLEX
TRAIT_SCALE = 1.1        # trait name, top-left of each SIDE cell
VIEW_SCALE  = 1.1        # SIDE / BACK, bottom-left of every cell
THICK       = 2
PAD         = 18         # inset from the cell edge

# Vignette
KEY_BG    = [0.0, 1.0, 0.0]   # key green, absent from bone and landmarks
KEY_TOL   = 60                # masking tolerance for the key colour
VIG_INNER = 0.18              # fraction of cell half-diagonal that stays pure white
VIG_OUTER = 0.95              # fraction at which the trait colour reaches full opacity

# Render geometry 
WIDTH, HEIGHT = 640, 480
ROT_MATRIX    = o3d.geometry.get_rotation_matrix_from_axis_angle([0, 0, np.deg2rad(13)])
LM_RADIUS_FRAC = 0.022      # sphere radius as a fraction of mesh bounding-box diagonal
LM_COLOR       = [0.85, 0.16, 0.16]     # landmark spheres
BONE_COLOR     = [0.82, 0.82, 0.82]     # vertebra surface
RENDERERS = [o3d.visualization.rendering.OffscreenRenderer(WIDTH, HEIGHT) for _ in range(4)]

# Load config 
config_path = "model_params_config.json"
with open(config_path) as f:
    cfg = json.load(f)
print(f"\033[92mLoaded config from {config_path}\033[0m")

# Parse filenames
train_paths   = cfg["list_mesh_paths"]
all_vtk_files = [os.path.basename(f) for f in train_paths]
print(f"{len(all_vtk_files)} meshes listed in config")

In [ ]:
# Define functions

def _radial_ramp(h, w, inner=VIG_INNER, outer=VIG_OUTER):
    yy, xx = np.mgrid[0:h, 0:w]
    cy, cx = (h - 1) / 2, (w - 1) / 2
    d      = np.sqrt(((yy - cy) / cy) ** 2 + ((xx - cx) / cx) ** 2) / np.sqrt(2)
    t      = np.clip((d - inner) / (outer - inner), 0, 1)
    return t * t * (3 - 2 * t)          

def _vignette(cell, color, ramp, tol=KEY_TOL):
    key     = cell[2, 2].astype(int)              # corner is always background
    col_bgr = np.array(color[::-1]) * 255
    grad    = (255 * (1 - ramp[..., None]) + col_bgr * ramp[..., None]).astype(np.uint8)

    mask = (np.abs(cell.astype(int) - key) < tol).all(axis=2)
    out  = cell.copy()
    out[mask] = grad[mask]
    return out

def _build_mesh_with_landmarks(mesh_path, lm_path, deg=0):
    pv_mesh = pv.read(str(mesh_path))
    pv_mesh = pv_mesh.extract_surface(algorithm="dataset_surface").triangulate()
    pv_mesh = pv_mesh.compute_normals(cell_normals=False, point_normals=True,
                                      inplace=False, auto_orient_normals=True)
    bone = pv_to_o3d(pv_mesh)
    bone.compute_vertex_normals()
    bone.paint_uniform_color(BONE_COLOR)

    bbox   = bone.get_axis_aligned_bounding_box()
    diag   = float(np.linalg.norm(bbox.get_extent()))
    radius = diag * LM_RADIUS_FRAC

    coords, _ = load_mrk_json(lm_path)
    combined  = bone
    for pt in np.asarray(coords):
        sph = o3d.geometry.TriangleMesh.create_sphere(radius=radius, resolution=12)
        sph.translate(pt)
        sph.compute_vertex_normals()
        sph.paint_uniform_color(LM_COLOR)
        combined += sph

    combined.rotate(ROT_MATRIX, center=combined.get_center())
    if deg:
        R = o3d.geometry.get_rotation_matrix_from_axis_angle([0, 0, np.deg2rad(deg)])
        combined.rotate(R, center=combined.get_center())
    return combined, len(coords)           

def _text_color(bgr_cell):
    lum = bgr_cell[..., 0].mean() * 0.114 + \
          bgr_cell[..., 1].mean() * 0.587 + \
          bgr_cell[..., 2].mean() * 0.299
    return (40, 40, 40) if lum > 130 else (235, 235, 235)  

def _crop(panels, which):
    if which == "top_left":     return panels[:HEIGHT, :WIDTH]
    if which == "top_right":    return panels[:HEIGHT, WIDTH:]
    if which == "bottom_left":  return panels[HEIGHT:, :WIDTH]
    if which == "bottom_right": return panels[HEIGHT:, WIDTH:]
    raise ValueError(which)

# One color per trait, derived from that trait's marker
markers = ['P', '+', 's', 'd', 'X', 'o', '2']
colors  = [(0.65, 0.69, 0.12),   # pea soup
           (0.84, 0.65, 0.23),   # saffron
           (0.72, 0.44, 0.22),   # mud
           (0.36, 0.557, 0.68),  # powder blue
           (0.10, 0.51, 0.40),   # deep blue
           (0.60, 0.50, 0.46),   # slate
           (0, 0, 0)]            # black
marker_to_color = dict(zip(markers, colors))

In [ ]:
# Build specimen metadata and trait color table

pat = re.compile(r"^(?P<species>[\w\s\-]+)[\-_ ]+[\w\d]+[\-_ ]+(?P<vertebra>[CTL]?\d+)",
                 re.IGNORECASE)
parsed = [pat.match(f) for f in all_vtk_files]
specimens = pd.DataFrame({"mesh":        all_vtk_files,
                          "specimen_id": [m.group("species") if m else None for m in parsed],
                          "vertebra":    [m.group("vertebra").upper() if m else None for m in parsed]})
print(f"Parsed {specimens['specimen_id'].notna().sum()} / {len(specimens)} filenames")

sdf = pd.read_csv(SPECIES_CSV)
sdf["marker"] = sdf["marker"].astype(str).str.strip().str.strip("'\"")
specimens = specimens.merge(sdf, left_on="specimen_id", right_on="specimen", how="left")

unmatched = specimens.loc[specimens["family"].isna(), "specimen_id"].dropna().unique()
print(f"{specimens['family'].notna().sum()} / {len(specimens)} specimens matched to {SPECIES_CSV}")
if len(unmatched):
    print(f"\033[33mUnmatched specimen IDs ({len(unmatched)}):\033[0m {sorted(unmatched)[:10]}")

trait_marker = specimens.drop_duplicates("trait").set_index("trait")["marker"]
trait_colors = {t: marker_to_color.get(m, (0.5, 0.5, 0.5))
                for t, m in trait_marker.items() if pd.notna(t)}

unmapped = [t for t, m in trait_marker.items() if pd.notna(t) and m not in marker_to_color]
if unmapped:
    print(f"\033[33mTraits using a marker not in marker_to_color (grey): {unmapped}\033[0m")
print(f"{len(trait_colors)} traits: {sorted(trait_colors)}")

specimens.head()

## Render the panel

In [ ]:
# Make subplots for each life history strategy

mat = o3d.visualization.rendering.MaterialRecord()
mat.shader     = "defaultLit"
mat.base_color = [1.0, 1.0, 1.0, 1.0]     # white -- vertex colours carry the tint

for r in RENDERERS:
    r.scene.set_background(KEY_BG + [1.0])
blank = np.full((HEIGHT, WIDTH, 3), (np.array(KEY_BG) * 255).astype(np.uint8), dtype=np.uint8)

cells = {}
for trait, (specimen_id, vertebra) in EXEMPLARS.items():
    try:
        # mesh name comes from `specimens`; ids can contain hyphens so id+vertebra
        # don't concatenate back into the filename
        match = specimens[(specimens["specimen_id"].str.lower() == specimen_id.lower())
                          & (specimens["vertebra"].str.lower() == vertebra.lower())]
        if match.empty:
            raise KeyError(f"No row in `specimens` for ({specimen_id!r}, {vertebra!r})")
        mesh_path = Path(cfg["list_mesh_paths"][all_vtk_files.index(match.iloc[0]["mesh"])])

        for view_lbl, crop_key, deg in VIEWS:
            o3d_mesh, n_lms = _build_mesh_with_landmarks(
                mesh_path, LM_DIR / (mesh_path.stem + ".mrk.json"), deg)
            panels = render_cameras(RENDERERS, o3d_mesh, 0, mat, 1, n_rotations=1)
            cells[(view_lbl, trait)] = _crop(panels, crop_key).copy()
            del o3d_mesh, panels
            gc.collect()

        print(f"  {trait:<15s} {specimen_id}-{vertebra}  ({n_lms} landmarks)  OK", flush=True)

    except Exception as e:
        print(f"  {trait:<15s} FAILED: {e}")
        cells.update({(v[0], trait): blank for v in VIEWS})

In [ ]:
# Stack into  a panel figure for paper

traits = list(EXEMPLARS)
blocks = [traits[i:i + N_COLS] for i in range(0, len(traits), N_COLS)]
layout = [(block, view_lbl) for block in blocks for view_lbl, _, _ in VIEWS]

RAMP  = _radial_ramp(HEIGHT, WIDTH)       
panel = np.vstack([
    np.hstack([_vignette(cells[(view_lbl, t)], trait_colors.get(t, (0.5, 0.5, 0.5)), RAMP)
               for t in block])
    for block, view_lbl in layout])

for r, (block, view_lbl) in enumerate(layout):
    y0 = r * HEIGHT
    for j, trait in enumerate(block):
        x0  = j * WIDTH
        col = _text_color(panel[y0:y0 + HEIGHT, x0:x0 + WIDTH])
        if r % len(VIEWS) == 0:           
            cv2.putText(panel, trait.upper(), (x0 + PAD, y0 + PAD + 34),
                        FONT, TRAIT_SCALE, col, THICK, cv2.LINE_AA)
        cv2.putText(panel, view_lbl, (x0 + PAD, y0 + HEIGHT - PAD),
                    FONT, VIEW_SCALE, col, THICK, cv2.LINE_AA)

cv2.imwrite(str(OUT_PANEL), panel)
print(f"Saved -> {OUT_PANEL}   ({panel.shape[1]} x {panel.shape[0]} px)")

IPyImage(str(OUT_PANEL), width=900)